# mlreport Quickstart Template

## Imports

`Report` builds a report for one model. `ComparisonReport` compares two or more built reports.

In [ ]:
from mlreport import ComparisonReport, Report

## API Flow

Most reports follow the same sequence:

1. Create a `Report` around a fitted model.
2. Add evaluation data with `add_split()` or `add_crossval()`.
3. Optionally attach hyperparameter search results with `add_search()`.
4. Call `build()`.
5. Export the report.

### 1. Create a Report

For scikit-learn models, `mlreport` usually detects whether the model is for classification or regression. For custom models, pass `model_type` and `model_params` explicitly.

In [ ]:
report = Report(
    model,  # fitted sklearn-compatible or custom model
    title="Model Report",
    author="Your Name",
    description="Optional description of this model or experiment.",
    theme="light",  # or "dark"
    cmap="viridis",  # any matplotlib colormap
    model_type=None,  # optional: "classification" or "regression"
    model_params=None,  # optional dict of parameters to display
)

### 2. Add Evaluation Data

Choose one evaluation style: named splits or cross-validation. Note only one style is allowed per report.

#### Option A: Train/Test Splits

Use `add_split()` when you have one or more named datasets such as `train`, `validation`, or `test`.

If `y_pred` is not provided, `mlreport` calls `model.predict(X)`.

In [ ]:
report.add_split("train", X_train, y_train)
report.add_split("test", X_test, y_test)

# If you already computed predictions, pass them explicitly:
report.add_split("test", X_test, y_test, y_pred_test)

#### Option B: Cross-Validation

Use `add_crossval()` when you want one `cv` evaluation split built from out-of-fold predictions.

There are two choices:

- Let `mlreport` compute out-of-fold predictions with `cross_val_predict()`.
- Provide precomputed out-of-fold predictions yourself.

`cv` can be an integer fold count, a scikit-learn splitter, or an iterable of `(train_idx, test_idx)` pairs. `fold_ids` is a one-dimensional array with one fold label per row.

If `y_pred` is not provided, mlreport calls `cross_val_predict()`.

Do not pass both `cv` and `fold_ids` in the same call.

In [ ]:
# 1. Let mlreport compute out-of-fold predictions using an integer fold count.
report.add_crossval(X, y, cv=5)

# 2. Let mlreport compute predictions using a scikit-learn CV splitter.
report.add_crossval(X, y, cv=cv)

# 3. Let mlreport compute predictions using explicit train/test index pairs.
report.add_crossval(X, y, cv=splits)

# 4. Let mlreport compute predictions using one fold id per row.
report.add_crossval(X, y, fold_ids=fold_ids)

# 5. Provide precomputed out-of-fold predictions only.
# This gives aggregate CV metrics, but no fold-level summaries.
report.add_crossval(X, y, y_pred=y_pred_cv)

# 6. Provide precomputed predictions plus cv to include fold-level summaries.
report.add_crossval(X, y, y_pred=y_pred_cv, cv=cv)

# 7. Provide precomputed predictions plus explicit train/test index pairs.
report.add_crossval(X, y, y_pred=y_pred_cv, cv=splits)

# 8. Provide precomputed predictions plus one fold id per row.
report.add_crossval(X, y, y_pred=y_pred_cv, fold_ids=fold_ids)

### 3. Add Hyperparameter Search Results

If you used `GridSearchCV` or `RandomizedSearchCV`, attach the fitted search object before building the report.

The current tuning plots support exactly two tuned hyperparameters.

In [ ]:
report.add_search(search_cv)

### 4. Build Metrics and Plots

`build()` computes the metrics and creates the plots. You can exclude specific metrics or plots by ID.

Use `available_metrics()` and `available_plots()` to print the valid IDs for the current model type.

In [ ]:
report.available_metrics()
report.available_plots()

report.build(
    exclude_metrics=[],
    exclude_plots=[],
)

### 5. Export the Report

Reports can be exported to text, HTML, Markdown, JSON, or PDF.

In [ ]:
report.to_txt()
report.to_html("report.html")
report.to_md("report.md")
report.to_json("report.json")
report.to_pdf("report.pdf")

## Model Comparison

Build individual reports first, then pass them to `ComparisonReport`.

All reports in a comparison must use the same model type. The first report is treated as the baseline when deltas are shown.

In [ ]:
baseline_report = (
    Report(model_a, title="Baseline")
    .add_split("test", X_test, y_test)
    .build()
)

candidate_report = (
    Report(model_b, title="Candidate")
    .add_split("test", X_test, y_test)
    .build()
)

comparison = ComparisonReport(
    reports=[baseline_report, candidate_report],
    title="Model Comparison",
    split="test",
    theme="light",
).build()

comparison.to_html("comparison.html")